# Day 4 模块 5：残差分析

总分（R² / MAE）只说「整体怎样」。
**残差**看：哪些天偏得多、往哪边偏。


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day4_cafe_sales.csv'),
    Path('day4') / 'day4_cafe_sales.csv',
    Path('教学课程') / 'day4' / 'day4_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day4_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 准备特征 X 和目标 y（与 Day 3 同一套，方便对比）
# 特征列：可能影响营收的输入
feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp',
    'quality', 'competitors', 'holiday', 'location',
]
# 天气文字 → 数字分（晴最好，大雨最差）
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

X = df[feature_cols + ['weather_score']].copy()  # 特征表
y = df['sales'].copy()  # 目标：营收

print('特征列:', list(X.columns))
print('样本数:', len(X))
X.head()


In [ ]:
from sklearn.model_selection import train_test_split

# test_size=0.2：约 20% 当测试集；random_state=42：随机种子，结果可复现
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('训练集行数:', len(X_train))
print('测试集行数:', len(X_test))


## 1. 残差定义

```text
残差 = 预测 − 真实
```

- 残差 **> 0**：猜高了
- 残差 **< 0**：猜低了
- 理想情况：残差大多在 0 附近晃，没有明显「只会高估某一类天」


In [ ]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rf = RandomForestRegressor(
    n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

residual = y_pred - y_test.to_numpy()

print('残差平均（接近 0 较好）:', round(residual.mean(), 1))
print('残差标准差:', round(residual.std(), 1))
print('|残差| 平均 (=MAE):', round(np.abs(residual).mean(), 1))


## 2. 残差直方图

看整体是否大致堆在 0 附近。


In [ ]:
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(7, 4))
plt.hist(residual, bins=15, color='#6f4e37', edgecolor='white')
plt.axvline(0, color='red', linestyle='--', label='0')
plt.xlabel('残差 = 预测 − 真实')
plt.ylabel('天数')
plt.title('测试集残差分布')
plt.legend()
plt.tight_layout()
plt.show()


## 3. 真实 vs 预测散点

点越靠近对角线 y = x，猜得越准。


In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.7, color='#6f4e37')
lims = [
    min(y_test.min(), y_pred.min()),
    max(y_test.max(), y_pred.max()),
]
plt.plot(lims, lims, 'r--', label='完美预测线')
plt.xlabel('真实营收')
plt.ylabel('预测营收')
plt.title('真实 vs 预测')
plt.legend()
plt.tight_layout()
plt.show()


## 4. 误差最大的几天

把测试集按 |残差| 排序，看模型「最不会」的样本——可能是极端天气、极端竞争等盲区线索。


In [ ]:
err_df = X_test.copy()
err_df['真实营收'] = y_test.to_numpy()
err_df['预测营收'] = y_pred.round(1)
err_df['残差'] = residual.round(1)
err_df['绝对误差'] = np.abs(residual).round(1)

# 误差最大的 8 天
worst = err_df.sort_values('绝对误差', ascending=False).head(8)
worst


## 5. 读盲区时注意

- 样本少时，几天极端值就会进「最差榜」
- 不要看到一天猜错就否定整个模型
- 业务上可问：这些天是否有表里**没有的特征**（活动、修路、网红探店……）

## 6. 小练习

1. 残差直方图如果明显偏在右边，说明整体更容易怎样？
2. 从 worst 表里挑一天，用特征列描述：这是怎样的经营日？
3. 改进方向可以有哪些？（更多数据 / 新特征 / 换模型 / 调参）


In [ ]:
# 在这里写
